In [1]:
import sys
sys.path.insert(0, r"C:\Users\nasri\OneDrive\Desktop\Projects\SIS_Marches\backend")

In [2]:
from sqlalchemy import text
from app.core.database import engine

def build_url(ref, org):
    return f"https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation={ref}&orgAcronyme={org}"

with engine.connect() as conn:
    # Vérification du format sur des lignes déjà correctes (doit matcher exactement)
    rows = conn.execute(text(
        "SELECT ref_consultation, org_acronyme, url_avis FROM appel_offres WHERE url_avis IS NOT NULL LIMIT 5"
    )).fetchall()
    print("--- Vérification du format sur données existantes ---")
    for ref, org, url_reel in rows:
        url_genere = build_url(ref, org)
        print("MATCH" if url_genere == url_reel else "MISMATCH", "-", url_genere)

    # Prévisualisation pour les 7 lignes à corriger
    print("\n--- Prévisualisation pour les AO sans url_avis ---")
    rows_a_corriger = conn.execute(text(
        "SELECT id, ref_consultation, org_acronyme FROM appel_offres WHERE url_avis IS NULL"
    )).fetchall()
    for id_, ref, org in rows_a_corriger:
        print(id_, "->", build_url(ref, org))

--- Vérification du format sur données existantes ---
MATCH - https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation=1024693&orgAcronyme=o8p
MATCH - https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation=1017789&orgAcronyme=l1f
MATCH - https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation=1022621&orgAcronyme=w7t
MATCH - https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation=1017726&orgAcronyme=a1z
MATCH - https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation=1022616&orgAcronyme=w7t

--- Prévisualisation pour les AO sans url_avis ---


l update réel


In [3]:
from sqlalchemy import text
from app.core.database import engine

def build_url(ref, org):
    return f"https://www.marchespublics.gov.ma/index.php?page=entreprise.EntrepriseDetailConsultation&refConsultation={ref}&orgAcronyme={org}"

with engine.connect() as conn:
    rows = conn.execute(text(
        "SELECT id, ref_consultation, org_acronyme FROM appel_offres WHERE url_avis IS NULL"
    )).fetchall()

with engine.begin() as conn:
    for id_, ref, org in rows:
        conn.execute(
            text("UPDATE appel_offres SET url_avis = :url WHERE id = :id"),
            {"url": build_url(ref, org), "id": id_}
        )

print(f"{len(rows)} AO mis à jour.")

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM appel_offres")).scalar()
    non_null = conn.execute(text("SELECT COUNT(*) FROM appel_offres WHERE url_avis IS NOT NULL")).scalar()
    print(f"Vérification finale : {non_null}/{total} AO ont désormais une url_avis")

0 AO mis à jour.
Vérification finale : 226/226 AO ont désormais une url_avis


In [4]:
from sqlalchemy import text
from app.core.database import engine

with engine.connect() as conn:
    # remplace 5 par l'id réel du projet qui affiche ce message
    projet = conn.execute(text("SELECT appel_offres_id FROM projets WHERE id = 5")).first()
    print("appel_offres_id:", projet)
    if projet and projet[0]:
        print(conn.execute(text(
            "SELECT statut, erreur, date_analyse FROM analyse_dce WHERE appel_offres_id = :a"
        ), {"a": projet[0]}).first())

appel_offres_id: (129,)
('partielle', None, '2026-07-22 15:47:10')


In [2]:
import sys
sys.path.insert(0, r"C:\Users\nasri\OneDrive\Desktop\Projects\SIS_Marches\backend")

In [3]:
from sqlalchemy import text
from app.core.database import engine

with engine.connect() as conn:
    print(conn.execute(text(
        "SELECT statut, origine, COUNT(*) FROM projets GROUP BY statut, origine"
    )).fetchall())

[('en_execution', 'manuel', 1), ('interesse', 'appel_offres', 11), ('soumis', 'appel_offres', 1)]
